In [1]:
import requests
import zipfile

print("Githubからank.zipのダウンロードを開始します")

url = "https://raw.githubusercontent.com/aiknowledgelearning-decuments/document01/main/section10/ank.zip"
r = requests.get(url)

with open("ank.zip", "wb") as f:
    f.write(r.content)

print("ank.zip のダウンロードが完了しました")

# ZIPを解凍
with zipfile.ZipFile("ank.zip", "r") as zip_ref:
    zip_ref.extractall(".")

print("ank.zip の解凍が完了しました")

Githubからank.zipのダウンロードを開始します
ank.zip のダウンロードが完了しました
ank.zip の解凍が完了しました


In [2]:
import sqlite3

DB_PATH = "ank.db"

CATEGORY_NAME_MAP = {
    "policy": "政策・制度",
    "definition": "用語・定義",
    "fact": "事実・数値",
    "context": "会議文脈・進行情報",
    "noise": "ノイズ・不要情報"
}

TARGET_CATEGORIES = ["context", "noise"]
LIMIT_PER_CATEGORY = 10

conn = sqlite3.connect(DB_PATH)
conn.row_factory = sqlite3.Row
cursor = conn.cursor()

print("ナレッジ化対象外QA サンプル表示")
print("=================================")

for category in TARGET_CATEGORIES:

    category_jp = CATEGORY_NAME_MAP.get(category, category)

    cursor.execute("""
    SELECT qa_id,
           qa_category,
           qa_category_reason,
           is_knowledge_target,
           question,
           answer
    FROM qa
    WHERE qa_category = ?
      AND COALESCE(is_knowledge_target,0)=0
    ORDER BY qa_id
    LIMIT ?
    """, (category, LIMIT_PER_CATEGORY))

    rows = cursor.fetchall()

    print("")
    print(f"### {category} ({category_jp})")
    print(f"表示件数: {len(rows)}")
    print("---------------------------------")

    for idx, row in enumerate(rows, start=1):
        print(f"[{idx}] qa_id: {row['qa_id']}")
        print(f"判定理由: {row['qa_category_reason']}")
        print(f"Q: {row['question']}")
        print(f"A: {row['answer']}")
        print("---------------------------------")

conn.close()

ナレッジ化対象外QA サンプル表示

### context (会議文脈・進行情報)
表示件数: 10
---------------------------------
[1] qa_id: 996
判定理由: 会議の進行や発言内容の文脈確認に関するため
Q: 会議の開始にあたり、委員長が確認したことは何ですか？
A: 本件調査のための政府参考人の出席を求め、説明を聴取することについて異議がないかを確認しました。
---------------------------------
[2] qa_id: 997
判定理由: 会議の進行や発言内容の文脈確認に関するため
Q: 会議に関して異議があったかどうかはどうでしたか？
A: 〔「異議なし」と呼ぶ者あり〕として、異議はありませんでした。
---------------------------------
[3] qa_id: 998
判定理由: 会議の進行や確認方法に関する文脈情報のため
Q: 会議の決定はどのように確認されましたか？
A: 中村委員長が御異議なしと認めたため、決定しました。
---------------------------------
[4] qa_id: 999
判定理由: 会議の進行担当者に関する文脈確認のため
Q: 誰が会議の進行を担当していましたか？
A: 中村委員長が会議の進行を担当していました。
---------------------------------
[5] qa_id: 1000
判定理由: 会議の進行や文脈確認に関する内容のため
Q: 会議の決議に異議はありましたか？
A: 異議はありませんでした。
---------------------------------
[6] qa_id: 1001
判定理由: 会議の進行や決定方法の文脈確認に関する内容のため
Q: 今の決定はどのような形でなされましたか？
A: 「御異議なし」と認められたことにより決議されました。
---------------------------------
[7] qa_id: 1004
判定理由: 会議の進行方法に関する文脈確認のため
Q: 質疑はどのように進められますか？
A: 中村委員長が順次質疑を許可します。
---------------------------------
[8

In [4]:
import sqlite3
import json
import numpy as np

DB_PATH = "ank.db"

# 表示する代表QA数
REP_LIMIT = 10

# 代表QAごとに表示する類似QA数
SIMILAR_LIMIT = 5


def cosine_similarity(a, b):
    va = np.array(a, dtype=np.float32)
    vb = np.array(b, dtype=np.float32)

    na = np.linalg.norm(va)
    nb = np.linalg.norm(vb)

    if na == 0 or nb == 0:
        return 0.0

    return float(np.dot(va, vb) / (na * nb))


def load_vector(text):
    return json.loads(text)


conn = sqlite3.connect(DB_PATH)
conn.row_factory = sqlite3.Row
cursor = conn.cursor()

# =====================================
# 代表QAを取得
# 件数が多いクラスタを優先
# =====================================
cursor.execute("""
SELECT qa_id,
       cluster_id,
       cluster_size,
       question,
       answer,
       representative_score,
       embedding_vector
FROM qa
WHERE COALESCE(is_representative, 0) = 1
  AND COALESCE(is_knowledge_target, 0) = 1
  AND cluster_id IS NOT NULL
  AND embedding_vector IS NOT NULL
  AND embedding_vector <> ''
ORDER BY cluster_size DESC,
         representative_score DESC,
         cluster_id
LIMIT ?
""", (REP_LIMIT,))

representatives = cursor.fetchall()

print(f"表示対象の代表QA件数: {len(representatives)}")
print("=================================")

for rep_no, rep in enumerate(representatives, start=1):
    rep_qa_id = rep["qa_id"]
    cluster_id = rep["cluster_id"]
    cluster_size = rep["cluster_size"]
    rep_question = rep["question"] or ""
    rep_answer = rep["answer"] or ""
    rep_score = rep["representative_score"]

    try:
        rep_vector = load_vector(rep["embedding_vector"])
    except Exception as e:
        print(f"代表QAのembedding読込失敗 qa_id={rep_qa_id}: {e}")
        continue

    print(f"[代表QA {rep_no}]")
    print(f"cluster_id       : {cluster_id}")
    print(f"cluster_size     : {cluster_size}")
    print(f"qa_id            : {rep_qa_id}")
    print(f"represent_score  : {rep_score}")
    print(f"Q: {rep_question}")
    print(f"A: {rep_answer}")
    print("")
    print("  類似QA:")

    # =====================================
    # 同じクラスタ内の非代表QAを取得
    # =====================================
    cursor.execute("""
    SELECT qa_id,
           question,
           answer,
           embedding_vector
    FROM qa
    WHERE cluster_id = ?
      AND qa_id <> ?
      AND COALESCE(is_knowledge_target, 0) = 1
      AND COALESCE(is_embedded, 0) = 1
      AND embedding_vector IS NOT NULL
      AND embedding_vector <> ''
    ORDER BY qa_id
    """, (cluster_id, rep_qa_id))

    similar_rows = cursor.fetchall()

    similar_items = []

    for row in similar_rows:
        try:
            vector = load_vector(row["embedding_vector"])
            sim = cosine_similarity(rep_vector, vector)

            similar_items.append({
                "qa_id": row["qa_id"],
                "question": row["question"] or "",
                "answer": row["answer"] or "",
                "similarity": sim
            })

        except Exception as e:
            print(f"  embedding読込失敗 qa_id={row['qa_id']}: {e}")

    similar_items.sort(key=lambda x: x["similarity"], reverse=True)

    if not similar_items:
        print("    類似QAなし")
    else:
        for idx, item in enumerate(similar_items[:SIMILAR_LIMIT], start=1):
            print(f"    [{idx}] 類似度: {item['similarity']:.4f}")
            print(f"        qa_id: {item['qa_id']}")
            print(f"        Q: {item['question']}")
            print(f"        A: {item['answer']}")
            print("")

    print("=================================")

conn.close()

表示対象の代表QA件数: 10
[代表QA 1]
cluster_id       : 107
cluster_size     : 6
qa_id            : 2179
represent_score  : 0.9247207641601562
Q: 本日の会議で議題となっている法律案は何ですか？
A: 公立の義務教育諸学校等の教育職員の給与等に関する特別措置法等の一部を改正する法律案です。

  類似QA:
    [1] 類似度: 0.9391
        qa_id: 1829
        Q: 本日の議題は何ですか？
        A: 公立の義務教育諸学校等の教育職員の給与等に関する特別措置法等の一部を改正する法律案です。

    [2] 類似度: 0.9377
        qa_id: 1809
        Q: 今回議題に上がった法律案は何に関するものですか？
        A: 公立の義務教育諸学校等の教育職員の給与等に関する特別措置法等の一部を改正する法律案です。

    [3] 類似度: 0.9336
        qa_id: 1824
        Q: 今回の会議の主な議題は何ですか？
        A: 公立の義務教育諸学校等の教育職員の給与等に関する特別措置法等の一部を改正する法律案です。

    [4] 類似度: 0.9113
        qa_id: 2560
        Q: 何の法律案が議題にされていますか？
        A: 公立の義務教育諸学校等の教育職員の給与等に関する特別措置法等の一部を改正する法律案です。

    [5] 類似度: 0.9020
        qa_id: 2176
        Q: この日の会議に付した案件は何ですか？
        A: 政府参考人出頭要求に関する件と公立の義務教育諸学校等の教育職員の給与等に関する特別措置法等の一部を改正する法律案です。

[代表QA 2]
cluster_id       : 80
cluster_size     : 6
qa_id            : 1770
represent_score  : 0.9127864837646484
Q: 修学旅行の行き先や内容は誰が決定するのか？
